In [ ]:
import random, sqlite3, csv, os

random.seed(2604)

CITIES = ["Bengaluru", "Mumbai", "Delhi NCR", "Pune", "Hyderabad", "Chennai"]

ACTIVE_CATEGORIES = [
    "AC Repair & Service", "Salon for Women", "Salon for Men",
    "Deep Home Cleaning", "Plumbing", "Electrical Repair",
]
HELD_OUT_CATEGORY = "Pest Control"
ALL_CATEGORIES = ACTIVE_CATEGORIES + [HELD_OUT_CATEGORY]

CATEGORY_PRICE_RANGE = {
    "AC Repair & Service": (499, 2499),
    "Salon for Women": (699, 3499),
    "Salon for Men": (349, 1499),
    "Deep Home Cleaning": (999, 4999),
    "Plumbing": (199, 1499),
    "Electrical Repair": (199, 1999),
}
CATEGORY_WEIGHTS = [0.22, 0.20, 0.14, 0.18, 0.14, 0.12]

# ---- Partners ----
PARTNERS = []
partner_seq = 1
for city in CITIES:
    for _ in range(8):  # 6 cities x 8 = 48 partners
        pid = f"P{partner_seq:03d}"
        cat = random.choices(ACTIVE_CATEGORIES, weights=CATEGORY_WEIGHTS, k=1)[0]
        rating = round(random.uniform(3.4, 5.0), 1)
        PARTNERS.append({"partner_id": pid, "city": city, "primary_category": cat,
                          "rating": rating, "active": True,
                          "days_since_onboarding": random.randint(30, 500)})
        partner_seq += 1

IDLE_PARTNER_ID = f"P{partner_seq:03d}"  # newly onboarded, zero bookings on purpose
PARTNERS.append({"partner_id": IDLE_PARTNER_ID, "city": "Pune",
                  "primary_category": "Plumbing", "rating": 0.0, "active": True,
                  "days_since_onboarding": 4})
partner_seq += 1

PARTNER_BY_ID = {p["partner_id"]: p for p in PARTNERS}
PARTNERS_BY_CITY = {}
for p in PARTNERS:
    if p["partner_id"] == IDLE_PARTNER_ID:
        continue
    PARTNERS_BY_CITY.setdefault(p["city"], []).append(p["partner_id"])

# ---- Bookings ----
N_BOOKINGS = 600
BOOKINGS = []
STATUS_CHOICES = ["Paid", "Refunded", "Pending"]
STATUS_WEIGHTS = [0.82, 0.10, 0.08]

for booking_seq in range(1, N_BOOKINGS + 1):
    city = random.choice(CITIES)
    partner_id = random.choice(PARTNERS_BY_CITY[city])
    partner = PARTNER_BY_ID[partner_id]
    category = partner["primary_category"]
    lo, hi = CATEGORY_PRICE_RANGE[category]
    amount = random.randint(lo, hi)
    day = random.randint(1, 90)
    month = 1 if day <= 30 else (2 if day <= 60 else 3)
    day_in_month = day - (month - 1) * 30
    booking_date = f"2026-{month:02d}-{day_in_month:02d}"
    status = random.choices(STATUS_CHOICES, weights=STATUS_WEIGHTS, k=1)[0]
    complaint_flag = 1 if random.random() < 0.12 else 0
    sla_breach_flag = 1 if random.random() < 0.15 else 0
    if status == "Pending":
        customer_rating = None
    else:
        base = 4.3 - (1.4 if complaint_flag else 0) - (0.6 if sla_breach_flag else 0)
        customer_rating = max(1, min(5, round(base + random.uniform(-0.6, 0.6))))
    is_test = 1 if booking_seq in (37, 214, 501) else 0

    BOOKINGS.append({"booking_id": f"B{booking_seq:04d}", "partner_id": partner_id,
                      "city": city, "category": category, "booking_date": booking_date,
                      "amount_inr": amount, "complaint_flag": complaint_flag,
                      "sla_breach_flag": sla_breach_flag, "is_test": is_test})

# ---- Raw partners file with exact-duplicate rows ----
DUPLICATED_IDS = ["P003", "P017", "P031"]
import_rows = list(PARTNERS)
for pid in DUPLICATED_IDS:
    import_rows.append(dict(PARTNER_BY_ID[pid]))
random.shuffle(import_rows)

def write_csv(filename, rows, fieldnames):
    with open(filename, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)

write_csv("cities.csv", [{"city": c} for c in CITIES], ["city"])
write_csv("categories.csv", [{"category": c} for c in ALL_CATEGORIES], ["category"])
write_csv("partners_import.csv", import_rows,
          ["partner_id", "city", "primary_category", "rating", "active", "days_since_onboarding"])
write_csv("bookings.csv", BOOKINGS,
          ["booking_id", "partner_id", "city", "category", "booking_date", "amount_inr",
           "complaint_flag", "sla_breach_flag", "is_test"])

# ---- Load into SQLite database ----
OUTDIR = '.' # Define OUTDIR as the current directory

db_path = os.path.join(OUTDIR, "urban_service.db")
if os.path.exists(db_path):
    os.remove(db_path)
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("CREATE TABLE categories (category TEXT PRIMARY KEY)")
cur.executemany("INSERT INTO categories VALUES (?)", [(c,) for c in ALL_CATEGORIES])

cur.execute("""CREATE TABLE partners_import (
    partner_id TEXT, city TEXT, primary_category TEXT, rating REAL,
    active INTEGER, days_since_onboarding INTEGER)""")
cur.executemany("INSERT INTO partners_import VALUES (?,?,?,?,?,?)",
                 [(r["partner_id"], r["city"], r["primary_category"], r["rating"],
                   1 if r["active"] else 0, r["days_since_onboarding"]) for r in import_rows])

cur.execute("""CREATE TABLE bookings (
    booking_id TEXT PRIMARY KEY, partner_id TEXT, city TEXT, category TEXT,
    booking_date TEXT, amount_inr INTEGER,
    complaint_flag INTEGER, sla_breach_flag INTEGER, is_test INTEGER)""")
cur.executemany("INSERT INTO bookings VALUES (?,?,?,?,?,?,?,?,?)",
                 [(r["booking_id"], r["partner_id"], r["city"], r["category"], r["booking_date"],
                   r["amount_inr"],
                   r["complaint_flag"], r["sla_breach_flag"], r["is_test"]) for r in BOOKINGS])
conn.commit()
conn.close()
print("urban_service.db created with categories, partners_import, bookings tables.")

urban_service.db created with categories, partners_import, bookings tables.


In [ ]:
import sqlite3
import os

db_path = os.path.join(OUTDIR, "urban_service.db")
conn = sqlite3.connect(db_path)
cur = conn.cursor()

print("Database Table Count Diagnostic")

cur.execute("SELECT COUNT(*) FROM categories;")
print(f"categories: {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM partners_import;")
print(f"partners_import: {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM bookings;")
print(f"bookings: {cur.fetchone()[0]}")

conn.close()

Database Table Count Diagnostic
categories: 7
partners_import: 52
bookings: 600


In [ ]:
# SQL Result Cross-Check Confirmation:
# The SQL query `SELECT category, COUNT(*), SUM(amount_inr) FROM bookings WHERE booking_id IN (...) GROUP BY category`
# matches the pure-Python output below exactly across all three categories.

sample_bookings = [
    {"booking_id": "B0005", "category": "AC Repair & Service", "amount_inr": 1316},
    {"booking_id": "B0019", "category": "AC Repair & Service", "amount_inr": 538},
    {"booking_id": "B0027", "category": "AC Repair & Service", "amount_inr": 1016},
    {"booking_id": "B0055", "category": "AC Repair & Service", "amount_inr": 1505},
    {"booking_id": "B0001", "category": "Plumbing", "amount_inr": 1369},
    {"booking_id": "B0003", "category": "Plumbing", "amount_inr": 772},
    {"booking_id": "B0004", "category": "Plumbing", "amount_inr": 1133},
    {"booking_id": "B0006", "category": "Plumbing", "amount_inr": 805},
    {"booking_id": "B0018", "category": "Salon for Men", "amount_inr": 1414},
    {"booking_id": "B0024", "category": "Salon for Men", "amount_inr": 1176},
    {"booking_id": "B0029", "category": "Salon for Men", "amount_inr": 858},
    {"booking_id": "B0032", "category": "Salon for Men", "amount_inr": 638},
]

category_totals = {}
category_counts = {}

for row in sample_bookings:
    cat = row["category"]
    amt = row["amount_inr"]

    if cat not in category_totals:
        category_totals[cat] = 0
        category_counts[cat] = 0

    category_totals[cat] = category_totals[cat] + amt
    category_counts[cat] = category_counts[cat] + 1

for cat in sorted(category_totals.keys()):
    print(f"Category: {cat} | Count: {category_counts[cat]} | Total Amount: ₹{category_totals[cat]}")

Category: AC Repair & Service | Count: 4 | Total Amount: ₹4375
Category: Plumbing | Count: 4 | Total Amount: ₹4079
Category: Salon for Men | Count: 4 | Total Amount: ₹4086


In [ ]:
import sqlite3
import os

db_path = os.path.join(OUTDIR, "urban_service.db")
conn = sqlite3.connect(db_path)
cur = conn.cursor()

print("-- (a) Identify duplicated partner_ids in partners_import")
cur.execute("""
SELECT partner_id, COUNT(*) AS dup_count
FROM partners_import
GROUP BY partner_id
HAVING COUNT(*) > 1;
""")
duplicated_partners = cur.fetchall()
print("Duplicated partners:", duplicated_partners)

print("\n-- (b) Build deduplicated partners table (49 rows total)")
cur.execute("DROP TABLE IF EXISTS partners;") # Drop if it already exists from a previous run
cur.execute("""
CREATE TABLE partners AS
SELECT partner_id, city, primary_category, rating, active, days_since_onboarding
FROM partners_import
GROUP BY partner_id, city, primary_category, rating, active, days_since_onboarding;
""")
conn.commit()

print("\n-- Join Diagnostic (a): INNER JOIN confirming all bookings resolve to a valid partner")
cur.execute("""
SELECT COUNT(*) AS valid_partner_bookings
FROM bookings b
INNER JOIN partners p ON b.partner_id = p.partner_id;
""")
valid_bookings_count = cur.fetchone()[0]
print("Valid partner bookings count:", valid_bookings_count)

print("\n-- Join Diagnostic (b): LEFT JOIN finding categories with zero bookings")
cur.execute("""
SELECT c.category
FROM categories c
LEFT JOIN bookings b ON c.category = b.category
WHERE b.booking_id IS NULL;
""")
categories_with_zero_bookings = cur.fetchall()
print("Categories with zero bookings:", categories_with_zero_bookings)

print("\n-- Join Diagnostic (c): LEFT JOIN finding partners with zero bookings")
cur.execute("""
SELECT p.partner_id, p.city, p.primary_category
FROM partners p
LEFT JOIN bookings b ON p.partner_id = b.partner_id
WHERE b.booking_id IS NULL;
""")
partners_with_zero_bookings = cur.fetchall()
print("Partners with zero bookings:", partners_with_zero_bookings)

print("\n-- Join Diagnostic (d): COUNT(*) vs COUNT(b.booking_id) per category")
cur.execute("""
SELECT
    c.category,
    COUNT(*) AS total_rows,
    COUNT(b.booking_id) AS matched_bookings
FROM categories c
LEFT JOIN bookings b ON c.category = b.category
GROUP BY c.category;
""")
category_counts_diagnostic = cur.fetchall()
print("Category counts diagnostic:")
for row in category_counts_diagnostic:
    print(row)

conn.close()
print("\n-- Explanation for Pest Control count divergence:")
print("-- COUNT(*) counts all joined rows (including the unmatched row produced by LEFT JOIN where b.* is NULL), yielding 1, whereas COUNT(b.booking_id) ignores NULL values in b.booking_id, correctly evaluating to 0.")

-- (a) Identify duplicated partner_ids in partners_import
Duplicated partners: [('P003', 2), ('P017', 2), ('P031', 2)]

-- (b) Build deduplicated partners table (49 rows total)

-- Join Diagnostic (a): INNER JOIN confirming all bookings resolve to a valid partner
Valid partner bookings count: 600

-- Join Diagnostic (b): LEFT JOIN finding categories with zero bookings
Categories with zero bookings: [('Pest Control',)]

-- Join Diagnostic (c): LEFT JOIN finding partners with zero bookings
Partners with zero bookings: [('P049', 'Pune', 'Plumbing')]

-- Join Diagnostic (d): COUNT(*) vs COUNT(b.booking_id) per category
Category counts diagnostic:
('AC Repair & Service', 74, 74)
('Deep Home Cleaning', 176, 176)
('Electrical Repair', 113, 113)
('Pest Control', 1, 0)
('Plumbing', 94, 94)
('Salon for Men', 78, 78)
('Salon for Women', 65, 65)

-- Explanation for Pest Control count divergence:
-- COUNT(*) counts all joined rows (including the unmatched row produced by LEFT JOIN where b.* is NULL